In [4]:
# EXPLICATION : Imports Evidently pour comparaison de distributions
# - Report : genere les rapports automatiques
# - DataDriftPreset : ensemble de metriques pour detecter le drift (Distribution, KS Test, etc.)
# - ColumnMapping : informe Evidently du type de chaque colonne (numerique/categorique)

import pandas as pd
import json
from pathlib import Path

try:
    from evidently.legacy.report import Report
    from evidently.legacy.metric_preset import DataDriftPreset
    from evidently.legacy.pipeline.column_mapping import ColumnMapping
except ImportError:
    # Fallback for older/newer Evidently layouts
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset
    from evidently.pipeline.column_mapping import ColumnMapping

print("✅ Evidently importe")


✅ Evidently importe


## Chargement référence et données production

In [5]:
# EXPLICATION : 
# 1. Référence = distribution d'entraînement (dataset pristine)
# 2. Production = features réelles extraites des logs d'inférence
# 3. Nettoyage : convertir "" en NaN (valeurs vides)
# 4. Aligner : garder seulement colonnes communes (peut y avoir des différences en production)

# Référence (entraînement)
reference = pd.read_csv("../reference/reference.csv")

# Production : extraire input_features des logs
LOG_FILE = Path("../logs/predictions.jsonl")
logs = pd.read_json(LOG_FILE, lines=True)
production = pd.json_normalize(logs['input_features'])

# Nettoyage ("" → NaN, aligner colonnes)
production = production.replace("", pd.NA).infer_objects()
# EXPLICATION : infer_objects() détecte automatiquement les vrais types (ex: strings → objects)

# Garder seulement les colonnes communes avec la référence
# (en production, certaines colonnes peuvent être absentes ou ajoutées)
common_cols = list(set(reference.columns) & set(production.columns))
reference = reference[common_cols]
production = production[common_cols]

# Supprimer les colonnes vides (100% NaN) pour éviter les erreurs Evidently
empty_ref = reference.columns[reference.isna().all()].tolist()
empty_prod = production.columns[production.isna().all()].tolist()
empty_cols = sorted(set(empty_ref) | set(empty_prod))
if empty_cols:
    reference = reference.drop(columns=empty_cols)
    production = production.drop(columns=empty_cols)
    print(f"⚠️ Colonnes vides supprimées : {len(empty_cols)}")

print(f"✅ Référence : {len(reference)} lignes | Production : {len(production)} lignes")
print(f"   Colonnes analysées : {len(reference.columns)}")

⚠️ Colonnes vides supprimées : 31
✅ Référence : 10000 lignes | Production : 500 lignes
   Colonnes analysées : 711


## Calcul du data drift + génération du rapport

In [7]:
# EXPLICATION : ColumnMapping aide Evidently à utiliser les bonnes métriques
# - Features numériques : test KS (Kolmogorov-Smirnov) pour comparaison de distributions
# - Features catégorique : test Chi-Squared pour comparer les fréquences

column_mapping = ColumnMapping()
column_mapping.numerical_features = reference.select_dtypes(include=['number']).columns.tolist()
column_mapping.categorical_features = reference.select_dtypes(include=['object', 'bool']).columns.tolist()

print(f"   Numériques : {len(column_mapping.numerical_features)} | Catégorique : {len(column_mapping.categorical_features)}")

# EXPLICATION : DataDriftPreset inclut :
# - Drift per column (KS test pour numériques, Chi2 pour catégories)
# - Dataset drift ratio
# - Détection automatique pour seuil default (0.95 confiance)
data_drift_report = Report(metrics=[DataDriftPreset()])
data_drift_report.run(reference_data=reference, current_data=production, column_mapping=column_mapping)

# Sauvegarde HTML (dashboard interactif)
REPORT_DIR = Path("../reports")
REPORT_DIR.mkdir(exist_ok=True)
report_path = REPORT_DIR / "data_drift_report.html"
data_drift_report.save_html(str(report_path))
print("✅ Rapport généré : reports/data_drift_report.html")

   Numériques : 580 | Catégorique : 131
✅ Rapport généré : reports/data_drift_report.html


## Alertes automatiques

In [9]:
# EXPLICATION : 
# - Extraire les résultats du rapport (dictionnaire structuré)
# - Seuil 0.3 : drift_score > 0.3 = **drift modéré à fort** (sensibilité équilibrée)
#   * 0.1-0.3 = léger (toléré)
#   * > 0.3 = alerte (intervention recommandée)
# - Ce seuil est a :  selon besoin métier (plus strict = plus d'alertes)

# Exemple d'alerte sur features qui driftent fortement
report_dict = data_drift_report.as_dict()
drift_summary = None
for metric in report_dict.get("metrics", []):
    result = metric.get("result", {})
    if "drift_by_columns" in result:
        drift_summary = result["drift_by_columns"]
        break

if drift_summary is None:
    sample_keys = [list(m.get("result", {}).keys()) for m in report_dict.get("metrics", [])[:3]]
    print("⚠️ Impossible de trouver 'drift_by_columns' dans le rapport Evidently")
    print(f"   Exemples de clés disponibles : {sample_keys}")
else:
    drifted_features = [col for col, info in drift_summary.items()
                        if info.get("drift_detected") and info.get("drift_score", 0) > 0.3]

    if len(drifted_features) > 0:
        print(f"🔴 ALERTE : Drift détecté sur {len(drifted_features)} features !")
        print(f"   Exemples : {drifted_features[:5]}")
        print("\n   📋 Recommandations : ")
        print("   - Vérifier source des données (anomalie/changement)") 
        print("   - Envisager réentraînement du modèle")
        print("   - Ajouter monitoring continu sur ces features")
    else:
        print("✅ Aucun drift majeur détecté")

print("\n📊 Ouvre le fichier reports/data_drift_report.html dans ton navigateur pour le dashboard complet")

🔴 ALERTE : Drift détecté sur 1 features !
   Exemples : ['AMT_INCOME_TOTAL']

   📋 Recommandations : 
   - Vérifier source des données (anomalie/changement)
   - Envisager réentraînement du modèle
   - Ajouter monitoring continu sur ces features

📊 Ouvre le fichier reports/data_drift_report.html dans ton navigateur pour le dashboard complet
